In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df["label"].value_counts())
print(df["language"].value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df[["label", "language"]], random_state=42
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "label"}))

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(train_dataset)
print(test_dataset[0].keys())

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../models/detection/logs",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

In [ ]:
import re

def get_template_signature(text):
    return re.sub(r'\d+|₹[\d,]+|SEBI/[A-Z]+/\d+/[A-Z]*\d+', 'X', text)[:50]

df["template_sig"] = df["text"].apply(get_template_signature)
unique_sigs = df["template_sig"].unique().tolist()
print(f"Unique template signatures: {len(unique_sigs)}")

train_sigs, test_sigs = train_test_split(unique_sigs, test_size=0.25, random_state=42)

train_df2 = df[df["template_sig"].isin(train_sigs)]
test_df2 = df[df["template_sig"].isin(test_sigs)]

print(f"Train: {len(train_df2)}, Test: {len(test_df2)}")
print(train_df2["label"].value_counts())
print(test_df2["label"].value_counts())

In [ ]:
train_dataset2 = Dataset.from_pandas(train_df2[["text", "label_id"]].rename(columns={"label_id": "label"}))
test_dataset2 = Dataset.from_pandas(test_df2[["text", "label_id"]].rename(columns={"label_id": "label"}))

train_dataset2 = train_dataset2.map(tokenize_function, batched=True)
test_dataset2 = test_dataset2.map(tokenize_function, batched=True)

model2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model2.to(device)

training_args2 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v2",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../models/detection/logs_v2",
    logging_steps=10,
    report_to="none"
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=train_dataset2,
    eval_dataset=test_dataset2,
    compute_metrics=compute_metrics
)

trainer2.train()

In [ ]:
test_examples = [
    ("Your investment matured today. Please log in to check your updated portfolio value and download your statement.", "genuine"),
    ("Hi, myself Rakesh from stock market department, your file has been selected for special bonus, please share your PAN and bank details to release fund", "phishing"),
    ("As per the notification issued this week, the settlement cycle for the derivative segment has been revised effective next month.", "genuine"),
    ("अगर आप 1 लाख अभी लगाते हैं तो 15 दिन में 3 लाख वापस मिलेगा, बस अपना OTP भेज दीजिये सर", "phishing"),
    ("आपका फोलियो स्टेटमेंट तैयार है, कृपया अपने पंजीकृत ईमेल पर जांचें।", "genuine"),
]

model3.eval()
for text, true_label in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    print(f"True: {true_label:10s} | Predicted: {pred_label:10s} | Text: {text[:60]}...")

In [ ]:
df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df.columns.tolist())
print(df["template_id"].nunique())
df.head()

In [ ]:
template_labels = df.drop_duplicates("template_id")[["template_id", "label"]]

train_ids, test_ids = train_test_split(
    template_labels["template_id"],
    test_size=0.25,
    random_state=42,
    stratify=template_labels["label"]
)

train_df3 = df[df["template_id"].isin(train_ids)]
test_df3 = df[df["template_id"].isin(test_ids)]

print(f"Train templates: {len(train_ids)}, Test templates: {len(test_ids)}")
print(f"Train rows: {len(train_df3)}, Test rows: {len(test_df3)}")
print("Train label distribution:")
print(train_df3["label"].value_counts())
print("Test label distribution:")
print(test_df3["label"].value_counts())

In [ ]:
train_encodings3 = tokenizer(
    train_df3["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)
test_encodings3 = tokenizer(
    test_df3["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

train_labels3 = label_encoder.transform(train_df3["label"]).tolist()
test_labels3 = label_encoder.transform(test_df3["label"]).tolist()

class PhishingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset3 = PhishingDataset(train_encodings3, train_labels3)
test_dataset3 = PhishingDataset(test_encodings3, test_labels3)

print(f"Train dataset size: {len(train_dataset3)}")
print(f"Test dataset size: {len(test_dataset3)}")

In [ ]:
model3 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args3 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v3",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v3",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer3 = Trainer(
    model=model3,
    args=training_args3,
    train_dataset=train_dataset3,
    eval_dataset=test_dataset3,
    compute_metrics=compute_metrics
)

In [ ]:
trainer3.train()

In [ ]:
import numpy as np

predictions_output = trainer3.predict(test_dataset3)
preds = np.argmax(predictions_output.predictions, axis=1)

pred_labels = label_encoder.inverse_transform(preds)
true_labels = label_encoder.inverse_transform(test_labels3)

import pandas as pd
results_df = pd.DataFrame({
    "text": test_df3["text"].tolist(),
    "true": true_labels,
    "pred": pred_labels
})

print(results_df["pred"].value_counts())
print()
print("Misclassified phishing (predicted genuine):")
print(results_df[(results_df["true"] == "phishing") & (results_df["pred"] == "genuine")]["text"].tolist())

In [ ]:
missed = results_df[(results_df["true"] == "phishing") & (results_df["pred"] == "genuine")]
print(f"Total missed: {len(missed)}")
for t in missed["text"]:
    print("-", t)
    print()

In [ ]:
false_positives = results_df[(results_df["true"] == "genuine") & (results_df["pred"] == "phishing")]
print(f"Total false positives: {len(false_positives)}")
for t in false_positives["text"]:
    print("-", t)

In [ ]:
hindi_phishing_test = results_df.merge(
    test_df3[["text", "template_id"]], on="text", how="left"
)
print(hindi_phishing_test[(hindi_phishing_test["true"] == "phishing")].groupby("template_id")["pred"].value_counts())

In [ ]:
df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df["template_id"].nunique())

template_labels = df.drop_duplicates("template_id")[["template_id", "label"]]

train_ids4, test_ids4 = train_test_split(
    template_labels["template_id"],
    test_size=0.25,
    random_state=42,
    stratify=template_labels["label"]
)

train_df4 = df[df["template_id"].isin(train_ids4)]
test_df4 = df[df["template_id"].isin(test_ids4)]

print(f"Train templates: {len(train_ids4)}, Test templates: {len(test_ids4)}")
print(f"Train rows: {len(train_df4)}, Test rows: {len(test_df4)}")
print("Train label distribution:")
print(train_df4["label"].value_counts())
print("Test label distribution:")
print(test_df4["label"].value_counts())

In [ ]:
train_encodings4 = tokenizer(
    train_df4["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)
test_encodings4 = tokenizer(
    test_df4["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

train_labels4 = label_encoder.transform(train_df4["label"]).tolist()
test_labels4 = label_encoder.transform(test_df4["label"]).tolist()

train_dataset4 = PhishingDataset(train_encodings4, train_labels4)
test_dataset4 = PhishingDataset(test_encodings4, test_labels4)

print(f"Train dataset size: {len(train_dataset4)}")
print(f"Test dataset size: {len(test_dataset4)}")

model4 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args4 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v4",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v4",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer4 = Trainer(
    model=model4,
    args=training_args4,
    train_dataset=train_dataset4,
    eval_dataset=test_dataset4,
    compute_metrics=compute_metrics
)

In [ ]:
trainer4.train()

In [ ]:
import gc
import torch

for var_name in ["model", "model2", "model3", "trainer", "trainer2", "trainer3"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

In [ ]:
model3 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args3 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v3",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v3",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer3 = Trainer(
    model=model3,
    args=training_args3,
    train_dataset=train_dataset3,
    eval_dataset=test_dataset3,
    compute_metrics=compute_metrics
)

trainer3.train()

In [ ]:
print(trainer3.state.best_model_checkpoint)
print(trainer3.state.best_metric)

In [ ]:
model3.eval()
for text, true_label in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    print(f"True: {true_label:10s} | Predicted: {pred_label:10s} | Text: {text[:60]}...")

In [ ]:
model3.save_pretrained("../models/detection/mbert_final")
tokenizer.save_pretrained("../models/detection/mbert_final")
print("Model saved.")

In [ ]:
print(df["template_id"].nunique())

In [ ]:
import os
print(os.path.exists("../models/detection/mbert_final"))
print(os.listdir("../models/detection/mbert_final"))

In [ ]:
import pandas as pd

real_df = pd.read_csv("../data/synthetic/text/genuine_real_excerpts.csv")
print(real_df.shape)
print(real_df.columns.tolist())
real_df.head()

In [ ]:
model3.eval()
correct = 0
misclassified = []

for text in real_df["text"]:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    if pred_label == "genuine":
        correct += 1
    else:
        misclassified.append(text)

accuracy = correct / len(real_df)
print(f"Correct: {correct} / {len(real_df)}")
print(f"Accuracy on real excerpts: {accuracy:.3f}")
print()
print(f"Misclassified count: {len(misclassified)}")

In [ ]:
print(misclassified[0])

In [ ]:
print(label_encoder.classes_)